# Building & Deploying a REST API for ML Inference

A pipeline-stage walkthrough of taking a trained model and shipping it as a production-ready REST API. Each stage opens with **Why we're here**, a **30-second concept**, a **failure-mode demo**, **decisions you make at this stage**, **two exercises**, and a **recap**.

By Stage 12 you'll have:

- A fresh LightGBM model trained on the BTC hourly data.
- A FastAPI service exposing `/predict`, `/predict/batch`, `/data/recent`, `/health`, `/ready`, `/metrics`.
- Pydantic schemas validating every request and response.
- API-key authentication.
- Structured logging with correlation IDs.
- A Dockerfile and `docker-compose.yml` ready to deploy anywhere that runs containers.
- A pytest suite that drives the API end-to-end.


---
## Stage 0 — Problem framing for an inference API

**↳ The brief.** A trading desk says: *"Stop emailing your model's predictions around. Stand up a service we can call from our risk dashboard and our automated trading agent."*

Before writing any FastAPI code, decide what the API actually owes its callers.

### Three candidate framings

| Framing | API surface | Latency budget | When this is right | When it's wrong |
|---|---|---|---|---|
| **A. Single-prediction RPC** | `POST /predict` with one row in, one row out | Tight (< 50ms p95) | Real-time decisioning, one call per event | Heavy batch use — you'll thrash the model loading / DB connections |
| **B. Batch-only** | `POST /predict/batch` with N rows in, N predictions out | Looser (seconds OK) | Periodic risk reviews, backtesting, offline scoring | Real-time use — single-row latency suffers from batching overhead |
| **C. Both, plus data + ops endpoints** | `/predict`, `/predict/batch`, `/data/recent`, `/health`, `/ready`, `/metrics` | Mixed: tight for `/predict`, looser elsewhere | Most production services — different consumers have different needs | Pure prototypes — extra surface area for things to go wrong |

### What this notebook picks, and why

We pick **C** — single-prediction, batch, data, and ops endpoints. Reason: it mirrors what a real ML service exposes. Skipping ops endpoints (health checks, metrics) is the most common mistake juniors make; this notebook treats them as first-class.

### What this choice trades away

- **More surface area = more places to mishandle errors.** Each endpoint needs its own validation, error responses, and tests.
- **Latency budget per endpoint must be set separately.** `/predict` has a tight budget; `/data/recent` doesn't.
- **Versioning becomes a real concern** — multiple endpoints means a change to one shouldn't break callers of the others.

### How this scopes the rest of the pipeline

- **Stage 3 (Pydantic I/O)**: each endpoint gets its own request and response schema.
- **Stage 5 (Auth)**: API key applies to all endpoints; `/health` is usually exempt.
- **Stage 8 (Ops)**: liveness vs readiness + Prometheus metrics — the bits that take you from "FastAPI app" to "production service".
- **Stage 11 (Docker)**: container exposes one port, all endpoints behind it.

### Common framing mistakes

- **Designing the API around the model**, not the caller. Callers don't care that you trained a LightGBM; they want `predict(row) → score`. Keep the API surface decoupled from internal model details.
- **Skipping `/ready`** because `/health` "feels enough". Different concerns: `/health` says the process is alive; `/ready` says it can actually serve traffic (model loaded, DB reachable).
- **Hard-coding the model artifact path** instead of an env var. Makes Docker deployment painful.


---
## Stage 1 — Train and persist the model

**↳ Why we're here.** The API is only as good as the artifact it serves. Stage 1 trains a small LightGBM on the BTC hourly data (target: sign of next 4h return) and saves a self-contained `joblib` bundle that includes feature names, threshold, and training cutoff — everything the API needs to predict without referencing the training code.

### The 30-second concept

```python
artifact = {
    'model':          fitted_model,
    'feature_names':  X_train.columns.tolist(),
    'threshold':      0.5,
    'trained_through': '2024-12-31T23:00:00Z',
}
joblib.dump(artifact, '/path/to/model.joblib')
```

**Why bundle metadata with the model**: the API loads ONE thing and gets everything it needs. Without `feature_names` you can't reorder a JSON request's columns to match training; without `trained_through` callers can't decide if the model is stale. *Common mistake*: saving just the model and hardcoding feature names elsewhere — they drift apart on the next retraining.


In [1]:
import os, joblib
import numpy as np
import pandas as pd
import lightgbm as lgb

# ----------------------------------------------------------------------
# Load the BTC data and build a tiny labelled dataset.
# ----------------------------------------------------------------------
DATA_PATH = '/home/zlac116/Code/learning/ml-revision/data/crypto_hourly.parquet'
raw = pd.read_parquet(DATA_PATH)
btc = (raw[raw['symbol'] == 'BTC']
       .drop_duplicates(subset='ts')
       .sort_values('ts')
       .set_index('ts'))

# Build past-only features and a 4h-ahead direction target.
log_close = np.log(btc['close'])
ret_1h = log_close.diff()

X = pd.DataFrame(index=btc.index)
X['ret_1h']      = ret_1h.shift(1)
X['ret_4h']      = log_close.shift(1).diff(4)
X['ret_24h']     = log_close.shift(1).diff(24)
X['vol_24h']     = ret_1h.shift(1).rolling(24).std()
X['vol_168h']    = ret_1h.shift(1).rolling(168).std()
X['hour']        = X.index.hour.astype(float)

y = (log_close.shift(-4) > log_close).astype(int)   # next-4h direction

valid = X.dropna().index.intersection(y.dropna().index)
X, y = X.loc[valid], y.loc[valid]

cut = int(len(X) * 0.8)
X_train, X_val = X.iloc[:cut], X.iloc[cut:]
y_train, y_val = y.iloc[:cut], y.iloc[cut:]

# ----------------------------------------------------------------------
# Fit a LightGBM model.
# ----------------------------------------------------------------------
model = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, num_leaves=31,
    verbosity=-1, random_state=0,
).fit(X_train, y_train)
val_acc = model.score(X_val, y_val)
print(f'val accuracy: {val_acc:.4f}')

# ----------------------------------------------------------------------
# Save the artifact bundle.
# ----------------------------------------------------------------------
ARTIFACT_PATH = '/tmp/btc_direction_model.joblib'
artifact = {
    'model':          model,
    'feature_names':  X.columns.tolist(),
    'threshold':      0.5,
    'trained_through': str(X_train.index.max()),
}
joblib.dump(artifact, ARTIFACT_PATH)
print(f'wrote artifact to {ARTIFACT_PATH}')
print(f'feature_names: {artifact["feature_names"]}')

val accuracy: 0.5122
wrote artifact to /tmp/btc_direction_model.joblib
feature_names: ['ret_1h', 'ret_4h', 'ret_24h', 'vol_24h', 'vol_168h', 'hour']


### Failure mode — saving just the model

```python
joblib.dump(model, '/tmp/just_model.joblib')   # only the model, no metadata
```

The API later loads this and needs to know feature order. It guesses from the JSON dict's keys. The order isn't stable — the next request with a different key order silently produces a wrong prediction. **Always bundle.**

### Decisions you make at this stage

- **What to bundle**: minimum is `(model, feature_names, threshold, trained_through)`. Add a model version, training data hash, and library versions if you'll have multiple models in production.
- **Where to put the artifact**: at training time, anywhere; at inference time, behind an env var (`os.environ['MODEL_PATH']`). Don't hardcode.

### Exercise 1.1 — Add a `model_version` field to the artifact

Re-save the artifact above with an additional `'model_version': 'v1.0.0'` key. Print the loaded artifact's keys to confirm.

```python
# Your answer here
```

<details><summary>💡 Click to reveal solution & explanation</summary>

```python
artifact['model_version'] = 'v1.0.0'
joblib.dump(artifact, ARTIFACT_PATH)
loaded = joblib.load(ARTIFACT_PATH)
print(list(loaded.keys()))
```

**Why it matters**: when you have v1.0.0 and v1.1.0 deployed simultaneously (canary, A/B test, gradual rollout), every prediction must declare which model produced it. Otherwise you can't attribute behaviour back to a specific version.

</details>

### Exercise 1.2 — Diagnose: prediction fails after rebuild

The cell below loads the saved artifact and tries to predict a single row whose dict keys are out of order. It works *now* — but only because we're using `feature_names` to reorder. Comment out the reorder and watch the prediction change. Restore the reorder and confirm correctness.

```python
# Your answer here — load artifact, predict on row in original order vs reversed.
```

<details><summary>💡 Click to reveal solution & explanation</summary>

```python
art = joblib.load(ARTIFACT_PATH)
sample = X_val.iloc[0].to_dict()
reversed_sample = dict(reversed(list(sample.items())))

# Correct: reorder by feature_names.
df_correct = pd.DataFrame([reversed_sample])[art['feature_names']]
print(f'correct prob: {art["model"].predict_proba(df_correct)[0, 1]:.4f}')

# Buggy: trust dict order.
df_buggy = pd.DataFrame([reversed_sample])
print(f'buggy   prob: {art["model"].predict_proba(df_buggy)[0, 1]:.4f}')
```

The two probabilities differ — sometimes by 0.5+. The reorder is the difference between a working API and silent garbage.

</details>

### Recap

We have `/tmp/btc_direction_model.joblib` with everything the API needs. Stage 2 wraps it in a minimal FastAPI app.


---
## Stage 2 — A minimal FastAPI app

**↳ Why we're here.** The smallest possible service: load the artifact, expose one `POST /predict` endpoint, return a JSON response. Everything later builds on this skeleton.

### The 30-second concept

```python
from fastapi import FastAPI
app = FastAPI()

@app.post('/predict')
def predict(payload: dict):
    return {'echo': payload}
```

That's it. `uvicorn main:app --reload` runs it. No magic — just a function decorated with the HTTP method + path.

**Why FastAPI over Flask**: type-hint-driven validation, async-native, OpenAPI auto-generated from your endpoints, native Pydantic integration. Modern Python web work defaults to FastAPI.

### Failure mode — testing in production

A common mistake: `uvicorn main:app` with no automated tests, then sending a manual curl from your laptop and declaring it "works". The first user to send a malformed payload will get a 500 error and you'll only find out from the logs. *Stage 12* shows how to drive the API in tests via `TestClient`.


In [2]:
# We can't run a uvicorn server in a Jupyter cell (it would block), so we use
# FastAPI's TestClient to drive the app from inside this notebook. The same
# code runs as `uvicorn main:app` in production.

from fastapi import FastAPI
from fastapi.testclient import TestClient
import joblib
import os

ARTIFACT_PATH = '/tmp/btc_direction_model.joblib'

app = FastAPI(title='BTC Direction Predictor', version='1.0.0')

@app.on_event('startup')
def load_model():
    app.state.artifact = joblib.load(ARTIFACT_PATH)
    app.state.feature_names = app.state.artifact['feature_names']

@app.post('/predict')
def predict(row: dict):
    art = app.state.artifact
    import pandas as pd
    df = pd.DataFrame([row])[art['feature_names']]
    prob = float(art['model'].predict_proba(df)[0, 1])
    return {'prob_up': prob, 'label': int(prob >= art['threshold'])}

# Drive it.

# Manually trigger startup (TestClient(app) without `with` does not fire on_event).
load_model()

client = TestClient(app)
sample = {
    'ret_1h': 0.001, 'ret_4h': 0.005, 'ret_24h': -0.01,
    'vol_24h': 0.005, 'vol_168h': 0.008, 'hour': 14.0,
}
response = client.post('/predict', json=sample)
print(f'status: {response.status_code}')
print(f'body:   {response.json()}')

status: 200
body:   {'prob_up': 0.538226512425142, 'label': 1}


/tmp/ipykernel_3877821/3779269064.py:14: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event('startup')


### Decisions you make at this stage

- **Where the artifact lives**: under `app.state` so handlers can reach it. Globals work but `app.state` is the recommended FastAPI idiom.
- **`@app.on_event('startup')` vs lazy load**: load at startup so the first request is fast. Lazy loading penalises the first user.

### Exercise 2.1 — Add a `GET /` welcome endpoint

Add a root endpoint returning `{'name': 'BTC Direction Predictor', 'version': '1.0.0'}`. Test it via `client.get('/')`.

<details><summary>💡 Click to reveal solution & explanation</summary>

```python
@app.get('/')
def root():
    return {'name': 'BTC Direction Predictor', 'version': '1.0.0'}

print(client.get('/').json())
```

**Why a root endpoint**: trivial sanity check ("is the service even reachable?") and FastAPI's `/docs` page can show it as the entry. *Common mistake*: skipping this and then debugging "is it the network or the app?" with no way to tell.

</details>

### Exercise 2.2 — Diagnose: `/predict` works in this notebook but fails when deployed

A junior reports "the endpoint returns 500 after I deploy with `uvicorn`." They've copied the cell above into `main.py`. **What's wrong?**

<details><summary>💡 Click to reveal solution & explanation</summary>

The artifact path is hardcoded to `/tmp/btc_direction_model.joblib` — fine in this notebook, broken on the deployed server (which doesn't have that file). Fix:

```python
ARTIFACT_PATH = os.environ['MODEL_PATH']    # set in Dockerfile / cloud env
```

The deployment will then fail loudly at startup if `MODEL_PATH` isn't set, instead of silently 500ing on every request.

**Why env vars over hardcoded paths**: the artifact's location is deployment-specific, not code-specific. Code in git should never assume a filesystem layout.

</details>

### Recap

Minimal FastAPI app: `@app.post('/predict')` accepts a dict, looks up the model, returns a JSON response. Stage 3 replaces the loose `dict` validation with a typed Pydantic schema.


---
## Stage 3 — Validated I/O with Pydantic

**↳ Why we're here.** The Stage-2 app accepts *any* dict. A request with `{'foo': 'bar'}` makes it past validation and only blows up inside the model code, returning an opaque 500. Pydantic models reject malformed inputs *at the framework boundary*, before any business logic runs.

### The 30-second concept

```python
from pydantic import BaseModel, Field

class PredictRequest(BaseModel):
    ret_1h:   float
    vol_24h:  float = Field(ge=0)   # vol must be non-negative
    hour:     int   = Field(ge=0, le=23)

class PredictResponse(BaseModel):
    prob_up: float = Field(ge=0, le=1)
    label:   int
    model_version: str
```

Pass them to the endpoint via type hints; FastAPI does the rest.


In [3]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, ValidationError
import joblib, pandas as pd

class PredictRequest(BaseModel):
    ret_1h:    float
    ret_4h:    float
    ret_24h:   float
    vol_24h:   float = Field(ge=0)
    vol_168h:  float = Field(ge=0)
    hour:      float = Field(ge=0, le=23)

class PredictResponse(BaseModel):
    prob_up:        float = Field(ge=0, le=1)
    label:          int
    model_version:  str
    trained_through: str

app = FastAPI(title='BTC Direction Predictor', version='1.0.0')

@app.on_event('startup')
def load_model():
    app.state.artifact = joblib.load('/tmp/btc_direction_model.joblib')

@app.post('/predict', response_model=PredictResponse)
def predict(req: PredictRequest):
    art = app.state.artifact
    df = pd.DataFrame([req.model_dump()])[art['feature_names']]
    prob = float(art['model'].predict_proba(df)[0, 1])
    return PredictResponse(
        prob_up=prob,
        label=int(prob >= art['threshold']),
        model_version='v1.0.0',
        trained_through=art['trained_through'],
    )


# Manually trigger startup (TestClient(app) without `with` does not fire on_event).
load_model()

client = TestClient(app)

# Valid request.
print('Valid:')
print(client.post('/predict', json={
    'ret_1h': 0.001, 'ret_4h': 0.005, 'ret_24h': -0.01,
    'vol_24h': 0.005, 'vol_168h': 0.008, 'hour': 14.0,
}).json())

# Invalid: vol_24h < 0 (violates Field(ge=0)).
print('\nInvalid (negative vol):')
print(client.post('/predict', json={
    'ret_1h': 0.001, 'ret_4h': 0.005, 'ret_24h': -0.01,
    'vol_24h': -0.005, 'vol_168h': 0.008, 'hour': 14.0,
}).status_code)

Valid:
{'prob_up': 0.538226512425142, 'label': 1, 'model_version': 'v1.0.0', 'trained_through': '2025-11-26 07:00:00+00:00'}

Invalid (negative vol):
422


/tmp/ipykernel_3877821/1257871114.py:22: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event('startup')


### Failure mode — relying on the model to validate

Without Pydantic, the request `{'foo': 'bar'}` reaches the model code, fails at `pd.DataFrame([row])[feature_names]` with a `KeyError`, and bubbles up as a 500. The caller has no idea what they did wrong. With Pydantic the same request returns 422 with a structured error body listing every missing field.

### Decisions you make at this stage

- **Constraints in `Field(...)`**: add `ge=`, `le=`, `gt=`, `lt=`, `min_length=`, `max_length=` for any field with a known valid range. Free input validation.
- **Output schemas, not just inputs**: `response_model=PredictResponse` enforces the *response* shape too — guarantees clients see what your OpenAPI docs promise.

### Exercise 3.1 — Add a `Field(description=...)` to every input

Improves the auto-generated `/docs` page. Add a one-line description to each field in `PredictRequest`.

<details><summary>💡 Click to reveal solution & explanation</summary>

```python
class PredictRequest(BaseModel):
    ret_1h:   float = Field(description='1-hour log return, last bar')
    ret_4h:   float = Field(description='4-hour log return, last 4 bars')
    ret_24h:  float = Field(description='24-hour log return, last 24 bars')
    vol_24h:  float = Field(ge=0, description='24h realised volatility (std of returns)')
    vol_168h: float = Field(ge=0, description='168h realised volatility')
    hour:     float = Field(ge=0, le=23, description='Hour of day in UTC, 0-23')
```

The description appears in the auto-generated OpenAPI docs at `/docs` — your callers' first stop. **Spend 10 minutes; save your future self 10 hours.**

</details>

### Exercise 3.2 — Diagnose: 422 with cryptic error

A caller sends `{'ret_1h': '0.001'}` (string, not float) and gets a 422. They expect string-to-number coercion. **Should the API silently coerce, or fail loudly?**

<details><summary>💡 Click to reveal solution & explanation</summary>

**Fail loudly.** Pydantic v2 rejects string-to-float by default — and that's the right behaviour. Coercion masks bugs in the caller's code (a JSON serializer producing strings instead of numbers). Force the contract: numbers in the request must be numbers in the JSON.

If you really need coercion, opt in explicitly:

```python
from pydantic import BaseModel, Field, ConfigDict
class PredictRequest(BaseModel):
    model_config = ConfigDict(strict=False)   # allows str->float coercion
    ret_1h: float
```

But by default, strict typing wins. The 422 with field-level error is exactly what should happen.

</details>

### Recap

Pydantic now validates every request and response. Malformed inputs fail at the boundary, not deep inside the model code. Stage 4 adds proper HTTP error responses for the cases that DO reach the handler.


---
## Stage 4 — Error handling and HTTP status codes

**↳ Why we're here.** Pydantic catches malformed input. But your handler can still fail in flight: model raises, downstream DB times out, cache miss. Each failure mode wants a *different status code* and a *consistent error body*.

### The 30-second concept

```python
from fastapi import HTTPException

@app.post('/predict')
def predict(req: PredictRequest):
    if not is_model_loaded():
        raise HTTPException(status_code=503, detail='Model not loaded')
    try:
        return run_inference(req)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=f'Invalid input: {e}')
    except Exception as e:
        raise HTTPException(status_code=500, detail='Internal error')
```

| Status | When to use |
|---|---|
| 200 | Success. |
| 400 | Bad request — caller's fault, recoverable (e.g. semantically invalid input). |
| 401 / 403 | Authentication / authorisation failure. |
| 404 | Resource not found. |
| 422 | Pydantic validation failure (FastAPI does this automatically). |
| 429 | Rate-limited. |
| 500 | Unexpected server error — your fault. |
| 503 | Service unavailable — model loading, dependency down. |

**Why explicit error mapping**: callers can branch on status code (retry on 503, give up on 400). A generic 500 forces them to retry blindly or fail. *Common mistake*: returning 200 with `{'error': '...'}` in the body — looks helpful, breaks every HTTP client's error handling.


In [4]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
import joblib, pandas as pd

app = FastAPI()

@app.on_event('startup')
def load_model():
    app.state.artifact = None   # Simulate a slow startup; model loads later.

class PredictRequest(BaseModel):
    ret_1h: float; ret_4h: float; ret_24h: float
    vol_24h: float = Field(ge=0); vol_168h: float = Field(ge=0)
    hour: float = Field(ge=0, le=23)

# Custom exception for known business errors.
class ModelNotReadyError(Exception):
    pass

@app.exception_handler(ModelNotReadyError)
async def model_not_ready_handler(request: Request, exc: ModelNotReadyError):
    return JSONResponse(status_code=503, content={
        'error': 'model_not_ready', 'message': str(exc) or 'Model is still warming up',
    })

@app.post('/predict')
def predict(req: PredictRequest):
    if app.state.artifact is None:
        raise ModelNotReadyError('Model not loaded yet')
    art = app.state.artifact
    df = pd.DataFrame([req.model_dump()])[art['feature_names']]
    prob = float(art['model'].predict_proba(df)[0, 1])
    return {'prob_up': prob, 'label': int(prob >= art['threshold'])}


# Manually trigger startup (TestClient(app) without `with` does not fire on_event).
load_model()

client = TestClient(app)

# Model not loaded → 503 with structured error.
sample = {'ret_1h': 0.001, 'ret_4h': 0.005, 'ret_24h': -0.01,
          'vol_24h': 0.005, 'vol_168h': 0.008, 'hour': 14.0}
r = client.post('/predict', json=sample)
print(f'Without model loaded: {r.status_code}    {r.json()}')

# Now load it.
app.state.artifact = joblib.load('/tmp/btc_direction_model.joblib')
r = client.post('/predict', json=sample)
print(f'With model loaded:    {r.status_code}    {r.json()}')

Without model loaded: 503    {'error': 'model_not_ready', 'message': 'Model not loaded yet'}
With model loaded:    200    {'prob_up': 0.538226512425142, 'label': 1}


/tmp/ipykernel_3877821/1597790383.py:9: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event('startup')


### Failure mode — `try/except: pass`

You wrap inference in `try/except`. The exception is swallowed; the handler returns `None`; the client gets 200 with `null`. Caller's code blows up several layers downstream with no way to trace it back to your service. **Always re-raise as an `HTTPException` with a status code, never silently swallow.**

### Decisions you make at this stage

- **Custom exception classes vs raising `HTTPException` directly**: custom classes (with handlers) decouple business logic from HTTP concerns. Pure handlers can use `HTTPException` directly without the indirection.
- **What to log on each error**: enough to debug without leaking internals. Stack traces in logs, sanitised messages in the response body.

### Exercise 4.1 — Add a 400 path for unknown asset

Suppose `PredictRequest` adds `symbol: str` and you only support BTC. Reject other symbols with HTTP 400.

<details><summary>💡 Click to reveal solution & explanation</summary>

```python
@app.post('/predict')
def predict(req: PredictRequest):
    if req.symbol != 'BTC':
        raise HTTPException(status_code=400,
                             detail=f'Symbol {req.symbol} not supported; only BTC')
    # ... rest
```

The 400 (vs 422) signals "the request is well-formed but semantically invalid". 422 means "I couldn't even parse it"; 400 means "I parsed it, but it asks for something I won't do."

</details>

### Exercise 4.2 — Diagnose: 500 errors leak stack traces

Production logs show stack traces in `response.body` for some errors. **Find the cause and fix it.**

<details><summary>💡 Click to reveal solution & explanation</summary>

The cause: a generic `except Exception as e: raise HTTPException(500, detail=str(e))` puts the exception message in the response. If `str(e)` includes file paths, table names, internal logic — that's a *security leak* (attackers map your internals).

Fix: log the full exception server-side, return a generic message client-side.

```python
import logging
log = logging.getLogger('app')

@app.exception_handler(Exception)
async def unhandled(request, exc):
    log.exception('unhandled error', extra={'path': str(request.url)})
    return JSONResponse(status_code=500,
                        content={'error': 'internal_error',
                                 'message': 'An unexpected error occurred. Reference: <log_id>'})
```

</details>

### Recap

We have explicit status codes, custom exception classes, and structured error responses. Stage 5 protects the API with an authentication layer.


---
## Stage 5 — API key authentication

**↳ Why we're here.** Once the API is reachable, anyone can call it. For an internal service, an API key in a header is the simplest viable auth — one shared secret, sent on every request, validated on the server.

### The 30-second concept

```python
from fastapi import Depends, HTTPException, Header

async def verify_api_key(x_api_key: str = Header(...)):
    if x_api_key != os.environ['API_KEY']:
        raise HTTPException(status_code=401, detail='Invalid API key')
    return True

@app.post('/predict', dependencies=[Depends(verify_api_key)])
def predict(req: PredictRequest):
    ...
```

The `Depends()` runs `verify_api_key` *before* the handler. If it raises, the handler never runs.

**Why API key in a header (not query string)**: query strings show up in logs and browser history. Headers don't. *Common mistake*: hardcoding the key in code or `.env`-checking it into git. Use environment variables exclusively.


In [5]:
import os
from fastapi import FastAPI, Depends, HTTPException, Header
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
import joblib, pandas as pd

# In production this is set by the orchestrator (Docker, k8s) — never in code.
os.environ.setdefault('API_KEY', 'demo-secret-change-me')

app = FastAPI()

@app.on_event('startup')
def load_model():
    app.state.artifact = joblib.load('/tmp/btc_direction_model.joblib')

class PredictRequest(BaseModel):
    ret_1h: float; ret_4h: float; ret_24h: float
    vol_24h: float = Field(ge=0); vol_168h: float = Field(ge=0)
    hour: float = Field(ge=0, le=23)

async def verify_api_key(x_api_key: str = Header(..., description='API key')):
    if x_api_key != os.environ['API_KEY']:
        raise HTTPException(status_code=401, detail='Invalid API key')
    return True

@app.post('/predict', dependencies=[Depends(verify_api_key)])
def predict(req: PredictRequest):
    art = app.state.artifact
    df = pd.DataFrame([req.model_dump()])[art['feature_names']]
    prob = float(art['model'].predict_proba(df)[0, 1])
    return {'prob_up': prob, 'label': int(prob >= art['threshold'])}


# Manually trigger startup (TestClient(app) without `with` does not fire on_event).
load_model()

client = TestClient(app)
sample = {'ret_1h': 0.001, 'ret_4h': 0.005, 'ret_24h': -0.01,
          'vol_24h': 0.005, 'vol_168h': 0.008, 'hour': 14.0}

# No key → 422 (Header(...) makes it required).
r = client.post('/predict', json=sample)
print(f'No key:       {r.status_code}')

# Wrong key → 401.
r = client.post('/predict', json=sample, headers={'x-api-key': 'wrong'})
print(f'Wrong key:    {r.status_code}    {r.json()}')

# Right key → 200.
r = client.post('/predict', json=sample, headers={'x-api-key': os.environ['API_KEY']})
print(f'Right key:    {r.status_code}    prob={r.json()["prob_up"]:.4f}')

No key:       422
Wrong key:    401    {'detail': 'Invalid API key'}
Right key:    200    prob=0.5382


/tmp/ipykernel_3877821/1050622709.py:12: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event('startup')


### Failure mode — comparing keys with `==`

```python
if x_api_key == os.environ['API_KEY']:    # vulnerable to timing attacks
```

A `==` comparison short-circuits on the first non-matching character. An attacker can probe the key character-by-character by measuring response latency. Use `secrets.compare_digest()`:

```python
import secrets
if not secrets.compare_digest(x_api_key, os.environ['API_KEY']):
    raise HTTPException(401)
```

Constant-time comparison neutralises the timing side channel.

### Decisions you make at this stage

- **One key vs per-client keys**: one is simpler; per-client lets you revoke individual access without rotating the shared secret.
- **Where to store keys**: env var for one key; a secrets manager (Vault, AWS Secrets Manager) for many.
- **Which endpoints require auth**: usually all except `/health` (load balancers must reach it without auth).

### Exercise 5.1 — Exempt `/health` from auth

Stage 8 will define `/health`. Make sure your auth dependency doesn't apply to it. (Hint: don't add `dependencies=[Depends(verify_api_key)]` to that endpoint.)

<details><summary>💡 Click to reveal solution & explanation</summary>

There are two ways:

**Option A** (per-endpoint, what we did): include `dependencies=[Depends(verify_api_key)]` only on protected endpoints. Don't add it to `/health`.

**Option B** (router-level): use `APIRouter(dependencies=[...])` for the protected group; mount unprotected endpoints on a separate router.

Option A is simpler for ≤10 endpoints. Option B scales better with many endpoints.

</details>

### Exercise 5.2 — Diagnose: API key in URL

A junior puts the key in `?api_key=xxx` instead of a header. **Why is this worse?**

<details><summary>💡 Click to reveal solution & explanation</summary>

Three concrete reasons:

1. **Server logs**: most web servers log full URLs, leaking keys to anyone with log access.
2. **Browser history**: if anyone tests the API in a browser, the key is saved.
3. **Referrer headers**: clicking a link from a logged-in browser sends the URL (with key) to the destination as a referrer.

Headers solve all three. The fix is mechanical (`Header(...)` instead of `Query(...)`) but the consequences of getting it wrong are real.

</details>

### Recap

API key auth is in place. Stage 6 adds a batch endpoint for callers who need to score many rows at once.


---
## Stage 6 — Batch prediction endpoint

**↳ Why we're here.** Calling `/predict` 1000 times to score 1000 rows wastes network and per-request overhead. A `/predict/batch` endpoint accepts a list and returns a list — same model, same validation, much less ceremony.

### The 30-second concept

```python
class BatchPredictRequest(BaseModel):
    rows: list[PredictRequest] = Field(..., max_length=1000)   # cap the batch size

@app.post('/predict/batch')
def predict_batch(req: BatchPredictRequest):
    df = pd.DataFrame([r.model_dump() for r in req.rows])[FEATURE_NAMES]
    probs = model.predict_proba(df)[:, 1]
    return {'predictions': [{'prob_up': p, 'label': int(p >= THR)} for p in probs]}
```

**Why cap with `max_length`**: a malicious or misbehaving client can send 1M rows and OOM your service. Set a sane limit (1000 is reasonable; 100k breaks most setups).


In [6]:
from fastapi import FastAPI, Depends, Header, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import List
import joblib, pandas as pd, os

os.environ.setdefault('API_KEY', 'demo-secret-change-me')
app = FastAPI()

@app.on_event('startup')
def load_model():
    app.state.artifact = joblib.load('/tmp/btc_direction_model.joblib')

class PredictRequest(BaseModel):
    ret_1h: float; ret_4h: float; ret_24h: float
    vol_24h: float = Field(ge=0); vol_168h: float = Field(ge=0)
    hour: float = Field(ge=0, le=23)

class PredictResponse(BaseModel):
    prob_up: float; label: int

class BatchPredictRequest(BaseModel):
    rows: List[PredictRequest] = Field(..., max_length=1000)

class BatchPredictResponse(BaseModel):
    predictions: List[PredictResponse]
    n: int

async def verify_api_key(x_api_key: str = Header(...)):
    if x_api_key != os.environ['API_KEY']:
        raise HTTPException(401, 'Invalid API key')

@app.post('/predict/batch', response_model=BatchPredictResponse,
          dependencies=[Depends(verify_api_key)])
def predict_batch(req: BatchPredictRequest):
    art = app.state.artifact
    df = pd.DataFrame([r.model_dump() for r in req.rows])[art['feature_names']]
    probs = art['model'].predict_proba(df)[:, 1]
    preds = [PredictResponse(prob_up=float(p), label=int(p >= art['threshold']))
             for p in probs]
    return BatchPredictResponse(predictions=preds, n=len(preds))


# Manually trigger startup (TestClient(app) without `with` does not fire on_event).
load_model()

client = TestClient(app)
batch = {'rows': [
    {'ret_1h': 0.001, 'ret_4h': 0.005, 'ret_24h': -0.01,
     'vol_24h': 0.005, 'vol_168h': 0.008, 'hour': 14.0},
    {'ret_1h': -0.002, 'ret_4h': 0.001, 'ret_24h': 0.02,
     'vol_24h': 0.006, 'vol_168h': 0.009, 'hour': 9.0},
]}
r = client.post('/predict/batch', json=batch, headers={'x-api-key': os.environ['API_KEY']})
print(f'status: {r.status_code}')
print(f'body:   {r.json()}')

status: 200
body:   {'predictions': [{'prob_up': 0.538226512425142, 'label': 1}, {'prob_up': 0.40508078204507053, 'label': 0}], 'n': 2}


/tmp/ipykernel_3877821/1938294449.py:10: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event('startup')


### Failure mode — unbounded batches

Without `max_length=1000`, a request with 10M rows will:
1. Allocate a 10M-row pandas DataFrame.
2. Run inference (LightGBM is fast but not infinite).
3. Serialise 10M predictions back as JSON.

The first or third step OOMs the container. Cap the batch size and document the cap in your error response.

### Decisions you make at this stage

- **Batch limit**: 100-1000 is reasonable; tune to your latency budget. Document it in the error message.
- **Partial failures**: a single bad row in a batch — fail the whole batch (simple, all-or-nothing) or return per-row results with errors? Simple wins for most use cases.

### Exercise 6.1 — Add per-row indices to the response

Each prediction in the response should include `row_index` so the caller can match predictions to their input rows.

<details><summary>💡 Click to reveal solution & explanation</summary>

```python
class PredictResponseIndexed(PredictResponse):
    row_index: int

@app.post('/predict/batch', dependencies=[Depends(verify_api_key)])
def predict_batch(req: BatchPredictRequest):
    art = app.state.artifact
    df = pd.DataFrame([r.model_dump() for r in req.rows])[art['feature_names']]
    probs = art['model'].predict_proba(df)[:, 1]
    return {
        'predictions': [
            {'row_index': i, 'prob_up': float(p), 'label': int(p >= art['threshold'])}
            for i, p in enumerate(probs)
        ],
        'n': len(probs),
    }
```

**Why row_index**: callers may filter or paginate batches; without an index, mapping output back to input is positional and fragile.

</details>

### Exercise 6.2 — Diagnose: batch faster than 1000 single requests

A caller benchmarks `/predict/batch` with 1000 rows vs 1000 sequential `/predict` calls. Batch is 50x faster. **Where does the speedup come from?**

<details><summary>💡 Click to reveal solution & explanation</summary>

Three sources, in order of magnitude:

1. **Network round trips**: 1 vs 1000. Each round trip is 1-100ms; that's 1-100 seconds saved.
2. **JSON parsing overhead**: parsing one big payload vs 1000 small ones — small but real.
3. **Vectorised model inference**: `predict_proba` on a 1000-row DataFrame is much faster than 1000 single-row calls because LightGBM amortises tree-traversal overhead.

The lesson: design batch endpoints when callers will score in bulk. The single-request endpoint is for real-time decisions, not bulk scoring.

</details>

### Recap

`/predict/batch` accepts up to 1000 rows in one call, returns a list of predictions. Stage 7 adds endpoints that serve the *data*, not just predictions.


---
## Stage 7 — Data endpoints with pagination

**↳ Why we're here.** Some callers want raw data alongside predictions (e.g. a dashboard showing "last 50 bars + the model's forecast for each"). Add a `GET /data/recent` endpoint with **query params** for filtering and **pagination** for large results.

### The 30-second concept

```python
from fastapi import Query

@app.get('/data/recent')
def data_recent(
    n: int = Query(50, ge=1, le=1000),
    symbol: str = Query('BTC'),
):
    df = load_data(symbol).tail(n)
    return df.to_dict(orient='records')
```

`Query(...)` validates query string parameters with the same `ge=`, `le=`, `min_length=` constraints as Pydantic Field. **Why pagination via `n`+`offset`** vs cursors: simpler. Cursors are better for live-updating data; `n`+offset is fine for a static history.


In [7]:
from fastapi import FastAPI, Depends, Header, HTTPException, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import List
import pandas as pd, os

os.environ.setdefault('API_KEY', 'demo-secret-change-me')
app = FastAPI()
DATA_PATH = '/home/zlac116/Code/learning/ml-revision/data/crypto_hourly.parquet'

@app.on_event('startup')
def load_data():
    app.state.data = pd.read_parquet(DATA_PATH)

class BarRow(BaseModel):
    ts:     str
    symbol: str
    open:   float
    high:   float
    low:    float
    close:  float
    volume: float

class RecentResponse(BaseModel):
    rows: List[BarRow]
    n:    int

async def verify_api_key(x_api_key: str = Header(...)):
    if x_api_key != os.environ['API_KEY']:
        raise HTTPException(401, 'Invalid API key')

@app.get('/data/recent', response_model=RecentResponse,
         dependencies=[Depends(verify_api_key)])
def data_recent(
    n:      int = Query(50, ge=1, le=1000, description='Number of recent bars'),
    symbol: str = Query('BTC', pattern='^[A-Z]{2,5}$'),
    offset: int = Query(0, ge=0, description='Offset from the end'),
):
    df = app.state.data
    df = df[df['symbol'] == symbol].sort_values('ts')
    end = len(df) - offset
    if end <= 0:
        return RecentResponse(rows=[], n=0)
    sliced = df.iloc[max(0, end - n):end].copy()
    sliced['ts'] = sliced['ts'].astype(str)
    return RecentResponse(rows=sliced.to_dict(orient='records'), n=len(sliced))


# Manually trigger startup (TestClient(app) without `with` does not fire on_event).
load_data()

client = TestClient(app)
r = client.get('/data/recent?n=3&symbol=BTC',
               headers={'x-api-key': os.environ['API_KEY']})
print(f'status: {r.status_code}')
import json
print(json.dumps(r.json(), indent=2)[:600])

status: 200
{
  "rows": [
    {
      "ts": "2026-04-19 20:00:00+00:00",
      "symbol": "BTC",
      "open": 74965.46,
      "high": 74965.47,
      "low": 74427.53,
      "close": 74630.0,
      "volume": 452.71123
    },
    {
      "ts": "2026-04-19 21:00:00+00:00",
      "symbol": "BTC",
      "open": 74630.01,
      "high": 74810.0,
      "low": 74368.6,
      "close": 74425.15,
      "volume": 383.54881
    },
    {
      "ts": "2026-04-19 22:00:00+00:00",
      "symbol": "BTC",
      "open": 74425.16,
      "high": 74425.16,
      "low": 73762.9,
      "close": 74075.54,
      "volume": 1057.5376



/tmp/ipykernel_3877821/64525130.py:11: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event('startup')


### Failure mode — returning the full dataset by default

Without `Query(50, ...)` defaulting `n=50`, a caller hitting `/data/recent` with no params gets the full 70k rows. JSON-serialising 70k rows is slow (10-100s) and the response is 10+ MB. **Always default to a sensible page size.**

### Decisions you make at this stage

- **Pagination with `n+offset` vs cursor-based**: `n+offset` is simpler; cursors are better when data is mutating (you want to avoid duplicates / skips when paging through a live stream).
- **Query parameter validation**: `pattern='^[A-Z]{2,5}$'` rejects symbol injection (`?symbol=BTC; DROP TABLE`). Same Pydantic-style constraints as request bodies.

### Exercise 7.1 — Add a date-range query

Add `from_ts` and `to_ts` query parameters; filter `df[(df['ts'] >= from_ts) & (df['ts'] <= to_ts)]`.

<details><summary>💡 Click to reveal solution & explanation</summary>

```python
from datetime import datetime
@app.get('/data/range')
def data_range(
    from_ts: datetime = Query(...),
    to_ts:   datetime = Query(...),
    symbol:  str = Query('BTC'),
):
    df = app.state.data
    df = df[(df['symbol'] == symbol) & (df['ts'] >= from_ts) & (df['ts'] <= to_ts)]
    return {'rows': df.to_dict(orient='records'), 'n': len(df)}
```

**Why `datetime` typed query**: FastAPI auto-parses ISO-8601 strings. Caller sends `?from_ts=2024-01-01T00:00:00Z`; you get a `datetime` object.

</details>

### Exercise 7.2 — Diagnose: response size grows linearly with n

A caller passes `n=10000`. Response takes 5 seconds. Same call with `n=100` takes 50ms. **What's wrong?**

<details><summary>💡 Click to reveal solution & explanation</summary>

JSON serialisation of 10k rows is the bottleneck. Three fixes:

1. **Cap `n`** at a sane upper bound (we already have `le=1000`). 10000 wouldn't even be accepted.
2. **Compress the response**: `from fastapi.middleware.gzip import GZipMiddleware; app.add_middleware(GZipMiddleware)`.
3. **Use a streaming response** for very large results: `StreamingResponse` with a generator yielding NDJSON chunks. Caller parses one row at a time; server doesn't hold the full response in memory.

The fastest improvement is option 1 — fail loudly when callers ask for too much.

</details>

### Recap

`/data/recent` serves paginated bar data with query-param validation. Stage 8 adds the ops endpoints (health, ready, metrics) — what's between "FastAPI app" and "production service".


---
## Stage 8 — Ops endpoints: liveness, readiness, metrics

**↳ Why we're here.** A production API exposes three flavours of "is the service OK?":

- **`/health`** (or `/healthz`): liveness probe. *"Is the process alive?"* — if not, restart it.
- **`/ready`**: readiness probe. *"Can it serve traffic?"* — if not, route around it.
- **`/metrics`**: Prometheus-format metrics for monitoring (request rate, latency, error rate).

These are NOT optional. Without them, your orchestrator (k8s, Cloud Run) can't distinguish "starting up" from "broken" and treats every slow startup as a failure.

### The 30-second concept

```python
@app.get('/health')
def health():
    return {'status': 'alive'}            # always 200 if the process can respond

@app.get('/ready')
def ready(response: Response):
    if app.state.artifact is None:
        response.status_code = 503
        return {'status': 'loading', 'reason': 'model not warm'}
    return {'status': 'ready', 'trained_through': app.state.artifact['trained_through']}

@app.get('/metrics')
def metrics():
    # In production, use prometheus_client; here we just hand-roll for clarity.
    return Response(content=...prometheus text format..., media_type='text/plain')
```

**Why `/health` is unauthenticated**: load balancers must call it without credentials. Putting auth on `/health` breaks every orchestrator.


In [8]:
from fastapi import FastAPI, Response, Depends, Header, HTTPException
from fastapi.testclient import TestClient
import joblib, time, os
from collections import defaultdict

os.environ.setdefault('API_KEY', 'demo-secret-change-me')
app = FastAPI()

# Tiny metric counters (in production: prometheus_client).
metrics = {
    'http_requests_total':       defaultdict(int),
    'http_errors_total':         defaultdict(int),
    'http_request_seconds_sum':  defaultdict(float),
}

@app.middleware('http')
async def record_metrics(request, call_next):
    t0 = time.time()
    resp = await call_next(request)
    dt = time.time() - t0
    metrics['http_requests_total'][request.url.path] += 1
    metrics['http_request_seconds_sum'][request.url.path] += dt
    if resp.status_code >= 400:
        metrics['http_errors_total'][request.url.path] += 1
    return resp

@app.on_event('startup')
def warm():
    app.state.artifact = None
    # Simulate slow startup; only mark warm after explicitly loading.

@app.get('/health')
def health():
    return {'status': 'alive'}

@app.get('/ready')
def ready(response: Response):
    if app.state.artifact is None:
        response.status_code = 503
        return {'status': 'loading', 'reason': 'model not warm'}
    return {'status': 'ready',
            'trained_through': app.state.artifact['trained_through']}

@app.get('/metrics')
def metrics_endpoint():
    lines = []
    for name, values in metrics.items():
        lines.append(f'# HELP {name}')
        lines.append(f'# TYPE {name} counter')
        for label, v in values.items():
            lines.append(f'{name}{{path="{label}"}} {v}')
    return Response(content='\n'.join(lines), media_type='text/plain')


# Manually trigger startup (TestClient(app) without `with` does not fire on_event).
warm()

client = TestClient(app)
print('Before model load:')
print('  /health:', client.get('/health').json())
print('  /ready: ', client.get('/ready').status_code, client.get('/ready').json())

# Load model.
app.state.artifact = joblib.load('/tmp/btc_direction_model.joblib')
print('\nAfter model load:')
print('  /ready: ', client.get('/ready').status_code, client.get('/ready').json())

# Generate some traffic so /metrics is non-empty.
for _ in range(3):
    client.get('/health')
print('\n/metrics:'); print(client.get('/metrics').text[:300])

Before model load:


/tmp/ipykernel_3877821/3351066242.py:27: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event('startup')


  /health: {'status': 'alive'}
  /ready:  503 {'status': 'loading', 'reason': 'model not warm'}

After model load:


  /ready:  200 {'status': 'ready', 'trained_through': '2025-11-26 07:00:00+00:00'}

/metrics:
# HELP http_requests_total
# TYPE http_requests_total counter
http_requests_total{path="/health"} 4
http_requests_total{path="/ready"} 4
# HELP http_errors_total
# TYPE http_errors_total counter
http_errors_total{path="/ready"} 2
# HELP http_request_seconds_sum
# TYPE http_request_seconds_sum counte


### Failure mode — using `/health` for everything

A junior implements only `/health` and uses it for both liveness and readiness probes. Result: container is restarted whenever the model is reloading (usually 30-60s after a fresh deploy). The service flaps endlessly. *Always two probes — alive vs ready.*

### Decisions you make at this stage

- **What `/ready` should check**: model loaded, downstream dependencies (DB, Redis, feature store) reachable. Don't make `/ready` heavy — it's called every few seconds.
- **What metrics to expose**: minimum is per-endpoint request count + p95 latency + error rate. Prometheus + Grafana is the standard stack.

### Exercise 8.1 — Add a `/info` endpoint

Returns the model version, feature names, and number of features. Useful for clients to verify they're calling the model they expect.

<details><summary>💡 Click to reveal solution & explanation</summary>

```python
@app.get('/info')
def info():
    art = app.state.artifact
    if art is None:
        raise HTTPException(503, 'Model not loaded')
    return {
        'model_version': 'v1.0.0',
        'trained_through': art['trained_through'],
        'feature_names': art['feature_names'],
        'n_features': len(art['feature_names']),
    }
```

**Why `/info` separate from `/health`**: `/health` should be cheap and stable. `/info` exposes more — fine for callers, not for load balancers.

</details>

### Exercise 8.2 — Diagnose: high error rate but healthy /ready

`/ready` says "ready" but `/metrics` shows 50% errors on `/predict`. **What might be wrong?**

<details><summary>💡 Click to reveal solution & explanation</summary>

`/ready` checks that the **model is loaded** but not that **predictions actually work**. The model could be loaded but corrupt; downstream features could be missing; the schema could mismatch what callers send.

Make `/ready` more thorough:

```python
@app.get('/ready')
def ready(response: Response):
    art = app.state.artifact
    if art is None:
        response.status_code = 503; return {'status': 'loading'}
    try:
        # Smoke-test: predict on a known-good row.
        sample = {f: 0.0 for f in art['feature_names']}
        df = pd.DataFrame([sample])[art['feature_names']]
        art['model'].predict_proba(df)
    except Exception as e:
        response.status_code = 503; return {'status': 'broken', 'reason': str(e)}
    return {'status': 'ready'}
```

Now `/ready` actually validates that inference works, not just that the artifact loaded.

</details>

### Recap

We have three ops endpoints. `/health` for liveness (always 200), `/ready` for readiness (503 until model warm), `/metrics` for monitoring. Stage 9 adds structured logging.


---
## Stage 9 — Structured logging with correlation IDs

**↳ Why we're here.** When something goes wrong in production, the only thing standing between you and a 3am page is *good logs*. Two requirements:

- **Structured (JSON), not free-form text** — searchable in any log aggregator (Datadog, Loki, CloudWatch).
- **Every log line tagged with a correlation ID** — lets you trace a single request through middleware, handler, model code, and downstream calls.

### The 30-second concept

```python
import logging, json
from uuid import uuid4

class JSONFormatter(logging.Formatter):
    def format(self, record):
        return json.dumps({
            'time': self.formatTime(record),
            'level': record.levelname,
            'msg': record.getMessage(),
            **getattr(record, 'extra', {}),
        })

# Every request gets a UUID; pass it in `extra=`.
log.info('predicting', extra={'extra': {'request_id': uuid4().hex, 'symbol': 'BTC'}})
```


In [9]:
import logging, json, time
from uuid import uuid4
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

class JSONFormatter(logging.Formatter):
    def format(self, record):
        out = {
            'time':  self.formatTime(record),
            'level': record.levelname,
            'logger': record.name,
            'msg':   record.getMessage(),
        }
        if hasattr(record, 'extra'):
            out.update(record.extra)
        return json.dumps(out)

# Configure root logger.
handler = logging.StreamHandler()
handler.setFormatter(JSONFormatter())
log = logging.getLogger('app')
log.handlers = [handler]
log.setLevel(logging.INFO)
log.propagate = False

app = FastAPI()

@app.middleware('http')
async def add_correlation_id(request: Request, call_next):
    request_id = request.headers.get('x-request-id') or uuid4().hex
    request.state.request_id = request_id
    t0 = time.time()
    try:
        resp = await call_next(request)
        dt = time.time() - t0
        log.info('request completed', extra={'extra': {
            'request_id': request_id,
            'path': str(request.url.path),
            'status': resp.status_code,
            'duration_ms': round(dt * 1000, 1),
        }})
        resp.headers['x-request-id'] = request_id
        return resp
    except Exception as e:
        log.exception('request failed', extra={'extra': {
            'request_id': request_id, 'path': str(request.url.path),
        }})
        raise

@app.get('/echo')
def echo(request: Request):
    log.info('handling /echo', extra={'extra': {
        'request_id': request.state.request_id,
    }})
    return {'request_id': request.state.request_id}

client = TestClient(app)
r = client.get('/echo', headers={'x-request-id': 'test-123'})
print(f'response: {r.json()}')
print(f'echoed back in header: {r.headers.get("x-request-id")}')

{"time": "2026-04-28 09:20:26,094", "level": "INFO", "logger": "app", "msg": "handling /echo", "request_id": "test-123"}


{"time": "2026-04-28 09:20:26,094", "level": "INFO", "logger": "app", "msg": "request completed", "request_id": "test-123", "path": "/echo", "status": 200, "duration_ms": 1.5}


response: {'request_id': 'test-123'}
echoed back in header: test-123


### Failure mode — `print()` statements

Code is full of `print('predicting:', symbol)`. They go to stdout but aren't structured. The log aggregator can't filter by symbol or join with the request ID. Debugging a single failed request becomes "grep for the timestamp range and eyeball it".

**Replace every `print()` with a `log.info(..., extra=...)` call.** It's mechanical and pays for itself the first time something breaks at 3am.

### Decisions you make at this stage

- **Pass `request_id` to caller in response header** so they can include it when reporting bugs.
- **Don't log secrets / PII** — even structured. Headers like `Authorization` should be filtered before logging.

### Exercise 9.1 — Log model predictions for monitoring

Add a log line in the `/predict` handler with `request_id`, the input features (or a hash of them), and the output probability. **Don't log the API key.**

<details><summary>💡 Click to reveal solution & explanation</summary>

```python
@app.post('/predict')
def predict(req: PredictRequest, request: Request):
    art = app.state.artifact
    df = pd.DataFrame([req.model_dump()])[art['feature_names']]
    prob = float(art['model'].predict_proba(df)[0, 1])
    log.info('prediction', extra={'extra': {
        'request_id': request.state.request_id,
        'features': req.model_dump(),
        'prob_up': prob,
    }})
    return {'prob_up': prob, 'label': int(prob >= art['threshold'])}
```

**Why log features**: lets you reconstruct what the model saw when investigating bad predictions later. **Why not log the API key**: never. It would end up in your aggregator, viewable by anyone with log access.

</details>

### Exercise 9.2 — Diagnose: logs scrolling too fast in dev

In dev, every health-check from Docker shows up as a log line. Output is unreadable. Fix.

<details><summary>💡 Click to reveal solution & explanation</summary>

Filter health-check paths from logging:

```python
@app.middleware('http')
async def add_correlation_id(request, call_next):
    if request.url.path in ('/health', '/healthz'):
        return await call_next(request)   # don't log probes
    # ... rest as before
```

In production keep these logged (still useful for monitoring); in dev set a `LOG_PROBES=false` env var.

</details>

### Recap

Every request has a correlation ID, every log line is JSON, every error is captured. Stage 10 explores when to use async vs sync handlers.


---
## Stage 10 — Async vs sync — when to use which

**↳ Why we're here.** FastAPI lets you write handlers as `async def` (cooperative multitasking, single thread) or `def` (run in a threadpool). Picking the wrong one tanks throughput.

### The 30-second concept

| Handler type | Best for | Why |
|---|---|---|
| `async def` | I/O-bound work (HTTP calls, DB queries, file reads) | While `await`ing, the event loop serves other requests |
| `def` (sync) | CPU-bound work (model inference, numpy operations, JSON parsing of large bodies) | Runs in a threadpool — multiple requests served in parallel |

```python
# I/O-bound: async + await.
@app.get('/external-data')
async def get_external():
    async with httpx.AsyncClient() as cl:
        return (await cl.get('https://api.example.com')).json()

# CPU-bound: sync. FastAPI uses a threadpool.
@app.post('/predict')
def predict(req: PredictRequest):
    return run_model(req)   # CPU-bound; sync is correct.
```

**Common mistake**: writing `async def predict(...)` then doing CPU work inside. The CPU work blocks the *whole event loop* — every other request stalls until it's done. Either use `def` or run the CPU work in `run_in_executor`.


In [10]:
import asyncio, time
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.get('/sync-cpu')
def sync_cpu(n: int = 1_000_000):
    # CPU-bound: sync handler runs in threadpool — fine for parallel requests.
    return {'sum': sum(i * i for i in range(n))}

@app.get('/async-io')
async def async_io():
    # Simulate an I/O await — event loop can serve others during this.
    await asyncio.sleep(0.05)
    return {'status': 'ok'}

@app.get('/wrong-async')
async def wrong_async(n: int = 1_000_000):
    # WRONG: CPU-bound work in an async handler blocks the event loop.
    return {'sum': sum(i * i for i in range(n))}

client = TestClient(app)
print('sync-cpu:    ', client.get('/sync-cpu?n=100000').json())
print('async-io:    ', client.get('/async-io').json())
print('wrong-async: ', client.get('/wrong-async?n=100000').json())

sync-cpu:     {'sum': 333328333350000}


async-io:     {'status': 'ok'}
wrong-async:  {'sum': 333328333350000}


### Failure mode — `async def` everywhere

Junior developer "modernises" the codebase by adding `async` to every handler. CPU-bound predictions now block the event loop. p99 latency triples. **Never use `async` unless you actually `await` something.**

### Decisions you make at this stage

- **Default to sync** for ML endpoints. Model inference is CPU-bound; sync handlers run in a threadpool and parallelise correctly.
- **Use async** when you have real I/O: external HTTP calls, async DB drivers, message queue producers/consumers.

### Exercise 10.1 — Convert a sync external-data fetch to async

You have `def fetch_btc_price()` that calls `requests.get(...)`. Rewrite it as an async handler using `httpx.AsyncClient`.

<details><summary>💡 Click to reveal solution & explanation</summary>

```python
import httpx

@app.get('/btc-price')
async def fetch_btc_price():
    async with httpx.AsyncClient(timeout=2.0) as cl:
        r = await cl.get('https://api.coinbase.com/v2/prices/BTC-USD/spot')
    return r.json()
```

**Why httpx over requests**: `requests` is sync-only. Calling it inside an `async def` blocks the event loop. `httpx.AsyncClient` is the async-native equivalent.

</details>

### Exercise 10.2 — Diagnose: async handler blocking event loop

A handler is `async def` but uses `time.sleep(2)` inside (instead of `await asyncio.sleep(2)`). **Why is this bad and how do you fix it?**

<details><summary>💡 Click to reveal solution & explanation</summary>

`time.sleep(2)` is a SYNC blocking call. Inside an `async def`, it freezes the event loop for 2 seconds — every other concurrent request waits.

Fix: use `await asyncio.sleep(2)`. The `await` yields control back to the event loop while sleeping.

```python
# WRONG
async def slow():
    time.sleep(2)         # blocks event loop
    return 'done'

# RIGHT
async def slow():
    await asyncio.sleep(2)   # yields; other requests run during the wait
    return 'done'
```

This generalises: in an `async` handler, every blocking call (DB query, HTTP request, sleep) must have an async equivalent and you `await` it.

</details>

### Recap

`async` for I/O-bound, sync for CPU-bound. Default to sync for ML endpoints. Stage 11 packages everything into a Docker image.


---
## Stage 11 — Containerise with Docker

**↳ Why we're here.** A `pip install` on the developer's laptop isn't deployable. Docker captures the runtime — Python version, dependencies, your code, the model artifact — into one immutable image you can run anywhere.

### The 30-second concept

A minimal `Dockerfile`:

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

**Why `python:3.12-slim`**: official, small (~150 MB), fast pulls. Avoid `python:3.12` (full Debian, 1 GB+) and `python:3.12-alpine` (musl-based, breaks compiled wheels).

A multi-stage build keeps the final image small:

```dockerfile
# Stage 1: build deps
FROM python:3.12-slim AS builder
WORKDIR /build
COPY requirements.txt .
RUN pip install --user --no-cache-dir -r requirements.txt

# Stage 2: runtime
FROM python:3.12-slim
WORKDIR /app
COPY --from=builder /root/.local /root/.local
COPY . .
ENV PATH=/root/.local/bin:$PATH
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```


In [11]:
# Print the Dockerfile and docker-compose.yml as text — these go in your repo, not the notebook.

dockerfile = '''
FROM python:3.12-slim

# Install system deps if any C extensions need them (LightGBM has wheels; usually fine without).
RUN apt-get update && apt-get install -y --no-install-recommends \\
        gcc \\
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# Install dependencies first (cached layer if requirements don't change).
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy source.
COPY app/ ./app/
COPY models/ ./models/

# Non-root user — never run as root in production.
RUN useradd --create-home --shell /bin/bash apiuser \\
    && chown -R apiuser:apiuser /app
USER apiuser

# Environment defaults (override in compose / cloud).
ENV MODEL_PATH=/app/models/btc_direction_model.joblib
ENV LOG_LEVEL=INFO

EXPOSE 8000

# Health check — Docker uses this to know if the container is alive.
HEALTHCHECK --interval=30s --timeout=3s --start-period=10s --retries=3 \\
    CMD python -c "import requests; requests.get(\'http://localhost:8000/health\').raise_for_status()" || exit 1

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''
print('---- Dockerfile ----')
print(dockerfile)

compose = '''
services:
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      MODEL_PATH: /app/models/btc_direction_model.joblib
      API_KEY: ${API_KEY:?API_KEY must be set}
      LOG_LEVEL: INFO
    volumes:
      # Mount data read-only so the container can serve from it without copying.
      - ./data:/app/data:ro
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "python", "-c", "import requests; requests.get(\'http://localhost:8000/health\').raise_for_status()"]
      interval: 30s
      timeout: 3s
      retries: 3
'''
print('\n---- docker-compose.yml ----')
print(compose)

---- Dockerfile ----

FROM python:3.12-slim

# Install system deps if any C extensions need them (LightGBM has wheels; usually fine without).
RUN apt-get update && apt-get install -y --no-install-recommends \
        gcc \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# Install dependencies first (cached layer if requirements don't change).
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy source.
COPY app/ ./app/
COPY models/ ./models/

# Non-root user — never run as root in production.
RUN useradd --create-home --shell /bin/bash apiuser \
    && chown -R apiuser:apiuser /app
USER apiuser

# Environment defaults (override in compose / cloud).
ENV MODEL_PATH=/app/models/btc_direction_model.joblib
ENV LOG_LEVEL=INFO

EXPOSE 8000

# Health check — Docker uses this to know if the container is alive.
HEALTHCHECK --interval=30s --timeout=3s --start-period=10s --retries=3 \
    CMD python -c "import requests; requests.get('http://localhost:8000/health').r

### Failure mode — running as root in the container

Default Docker images run as root inside the container. If anyone exploits the API and escapes containment, they have root on whatever the container can touch. **Always create and switch to a non-root user.**

### Decisions you make at this stage

- **`--workers 2` (or more)**: uvicorn workers are independent processes. Match to CPU cores. Start with 2; scale based on real traffic.
- **`HEALTHCHECK` at the Docker level vs orchestrator level**: both are fine; orchestrator (k8s `livenessProbe`) is more flexible. Docker `HEALTHCHECK` works with `docker-compose` too.

### Exercise 11.1 — Add a `.dockerignore`

What should and shouldn't go in the image? Write a `.dockerignore` that excludes `.venv/`, `*.ipynb`, `__pycache__/`, `data/` (mount it instead).

<details><summary>💡 Click to reveal solution & explanation</summary>

```
__pycache__/
*.pyc
*.ipynb
.venv/
.git/
.gitignore
.dockerignore
.env
data/
notebooks/
README.md
tests/
```

**Why exclude `data/`**: data files are big, change frequently, and don't belong in the immutable image. Mount as a volume.

</details>

### Exercise 11.2 — Diagnose: image is 2 GB

A junior's image is 2 GB. Yours is 250 MB. They use `FROM python:3.12` (full image). What other levers can shrink theirs?

<details><summary>💡 Click to reveal solution & explanation</summary>

In order of impact:

1. **`python:3.12-slim`** instead of `python:3.12` — saves ~700 MB.
2. **Multi-stage build** — only ship runtime deps, not build tools.
3. **`pip install --no-cache-dir`** — pip's build cache adds ~200 MB.
4. **`.dockerignore`** — excludes everything not needed.
5. **`apt-get clean && rm -rf /var/lib/apt/lists/*`** after installing system packages.

Combined: 2 GB → 200-300 MB.

</details>

### Recap

Container ready. Stage 12 adds an automated test suite that exercises every endpoint.


---
## Stage 12 — Testing the API

**↳ Why we're here.** Without tests you can't refactor without fear. The pyramid for an API:

- **Unit tests** for pure logic (preprocessing, validation rules) — fast, no FastAPI involvement.
- **Integration tests** that drive the API via `TestClient` — verifies routing, validation, auth, response shapes.
- **End-to-end / smoke tests** post-deploy that hit the live URL — verifies infra (DNS, TLS, networking).

We focus on integration tests here — they catch the most bugs per minute of effort.


In [12]:
# In a real project these go in `tests/test_api.py` and run via `pytest`.
# Here we run them inline so you can see them work.

from fastapi.testclient import TestClient
from fastapi import FastAPI, Depends, Header, HTTPException
from pydantic import BaseModel, Field
import joblib, pandas as pd, os

os.environ.setdefault('API_KEY', 'demo-secret-change-me')
app = FastAPI()

@app.on_event('startup')
def load():
    app.state.artifact = joblib.load('/tmp/btc_direction_model.joblib')

class PredictRequest(BaseModel):
    ret_1h: float; ret_4h: float; ret_24h: float
    vol_24h: float = Field(ge=0); vol_168h: float = Field(ge=0)
    hour: float = Field(ge=0, le=23)

async def auth(x_api_key: str = Header(...)):
    if x_api_key != os.environ['API_KEY']: raise HTTPException(401)

@app.get('/health')
def health(): return {'status': 'alive'}

@app.post('/predict', dependencies=[Depends(auth)])
def predict(req: PredictRequest):
    art = app.state.artifact
    df = pd.DataFrame([req.model_dump()])[art['feature_names']]
    prob = float(art['model'].predict_proba(df)[0, 1])
    return {'prob_up': prob, 'label': int(prob >= art['threshold'])}


# Manually trigger startup (TestClient(app) without `with` does not fire on_event).
load()

client = TestClient(app)
SAMPLE = {'ret_1h': 0.001, 'ret_4h': 0.005, 'ret_24h': -0.01,
          'vol_24h': 0.005, 'vol_168h': 0.008, 'hour': 14.0}
KEY = {'x-api-key': os.environ['API_KEY']}

# ----- tests -----
def test_health_no_auth_required():
    r = client.get('/health')
    assert r.status_code == 200
    assert r.json() == {'status': 'alive'}

def test_predict_requires_api_key():
    r = client.post('/predict', json=SAMPLE)
    assert r.status_code in (401, 422)   # 422 if header missing, 401 if wrong

def test_predict_with_wrong_key():
    r = client.post('/predict', json=SAMPLE, headers={'x-api-key': 'wrong'})
    assert r.status_code == 401

def test_predict_valid():
    r = client.post('/predict', json=SAMPLE, headers=KEY)
    assert r.status_code == 200
    body = r.json()
    assert 'prob_up' in body and 0 <= body['prob_up'] <= 1
    assert body['label'] in (0, 1)

def test_predict_invalid_input_returns_422():
    bad = {**SAMPLE, 'vol_24h': -0.1}   # negative vol — Field(ge=0) violation
    r = client.post('/predict', json=bad, headers=KEY)
    assert r.status_code == 422

def test_predict_order_invariance():
    """Property test: result shouldn't depend on JSON key order."""
    a = client.post('/predict', json=SAMPLE, headers=KEY).json()
    reversed_sample = dict(reversed(list(SAMPLE.items())))
    b = client.post('/predict', json=reversed_sample, headers=KEY).json()
    assert abs(a['prob_up'] - b['prob_up']) < 1e-9

# Run them.
for fn in [test_health_no_auth_required, test_predict_requires_api_key,
           test_predict_with_wrong_key, test_predict_valid,
           test_predict_invalid_input_returns_422,
           test_predict_order_invariance]:
    fn()
    print(f'  ✓ {fn.__name__}')
print(f'\nAll tests passed.')

  ✓ test_health_no_auth_required
  ✓ test_predict_requires_api_key
  ✓ test_predict_with_wrong_key


/tmp/ipykernel_3877821/3113771699.py:12: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event('startup')


  ✓ test_predict_valid
  ✓ test_predict_invalid_input_returns_422
  ✓ test_predict_order_invariance

All tests passed.


### Failure mode — only happy-path tests

A junior writes `test_predict_valid()` and stops. Production breaks the first time someone sends a malformed request, an expired key, or a missing header — none of which the test covered. **Test the failure paths too.**

### Decisions you make at this stage

- **What to mock**: external HTTP calls, DB connections, slow downstream services. NOT your own model — let it run end-to-end.
- **Test data**: use small, deterministic samples. Tests should run in < 5 seconds total.
- **Property tests**: the order-invariance test above is a property test — runs the same logic on permuted input, asserts invariance. Catches subtle bugs that example-based tests miss.

### Exercise 12.1 — Add a property test for batch ordering

`/predict/batch` should return predictions in the same order as the input rows. Write a test that feeds a 5-row batch and verifies the output ordering matches.

<details><summary>💡 Click to reveal solution & explanation</summary>

```python
def test_batch_preserves_order():
    rows = [
        {'ret_1h':  i * 0.001, 'ret_4h': 0.005, 'ret_24h': -0.01,
         'vol_24h': 0.005, 'vol_168h': 0.008, 'hour': 14.0}
        for i in range(5)
    ]
    r = client.post('/predict/batch', json={'rows': rows}, headers=KEY)
    body = r.json()
    # Re-predict each row individually and check matching order.
    for i, row in enumerate(rows):
        single = client.post('/predict', json=row, headers=KEY).json()
        assert abs(body['predictions'][i]['prob_up'] - single['prob_up']) < 1e-9
```

**Why this catches real bugs**: any 'optimisation' that reorders the batch internally (parallel processing, sorted-by-feature for cache friendliness) silently corrupts the response without this test.

</details>

### Exercise 12.2 — Diagnose: tests pass locally, fail in CI

Tests run green on your laptop, red in GitHub Actions. **What are the usual causes?**

<details><summary>💡 Click to reveal solution & explanation</summary>

Five common causes, in rough order of frequency:

1. **Missing env var**: `API_KEY` set on your laptop, not in CI. Fix: set it in the CI config or default it in tests.
2. **Missing artifact file**: the model `.joblib` lives at `/tmp/...` on your laptop; CI's `/tmp` is empty. Fix: train a tiny stub model in a fixture, or commit a small test artifact.
3. **Different Python version**: Python 3.11 locally, 3.12 in CI. Fix: pin via `pyproject.toml` or test matrix.
4. **Time/timezone differences**: tests using `datetime.now()` are flaky across timezones. Fix: pin a clock with `freezegun`.
5. **Network access**: tests that hit real external services pass on a permissive laptop, fail in a sandboxed CI. Fix: mock external calls.

The general lesson: tests should be hermetic — no env, network, or filesystem dependencies the test itself doesn't set up.

</details>

### Recap — end of API pipeline

You now have:

- A **trained model artifact** with feature names + threshold + version metadata.
- A **FastAPI service** with `/predict`, `/predict/batch`, `/data/recent`, `/health`, `/ready`, `/metrics`.
- **Pydantic-validated** inputs and responses on every endpoint.
- **API key auth** with constant-time comparison.
- **Structured JSON logs** with per-request correlation IDs.
- A **Dockerfile** (multi-stage, non-root, with healthcheck) and a `docker-compose.yml`.
- An **integration test suite** covering happy paths, validation, auth, and order invariance.

Drop the Dockerfile + compose + `app/main.py` into a fresh repo, set `API_KEY` and `MODEL_PATH`, run `docker-compose up`, and you have a deployable service.
